# finrag — SEC 10-K analysis pipeline

A walkthrough of the pipeline end to end. The logic lives in the `finrag`
package under `src/`; this notebook is the narrative, not the implementation.
That split is deliberate — the package is unit-tested, a notebook cell cannot be.

Install first, from the repository root:

```bash
pip install -e '.[local,semantic,google]'
cp .env.example .env      # then fill in SEC_CONTACT_EMAIL, and GOOGLE_API_KEY for the agent
```


In [ ]:
import logging

from dotenv import load_dotenv

from finrag.config import get_settings

load_dotenv()
logging.basicConfig(level=logging.INFO, format='%(levelname)-7s %(message)s')

settings = get_settings()
settings


## 1. Download filings from EDGAR

SEC requires a real contact address in the User-Agent of every request, read from
`SEC_CONTACT_EMAIL`. Downloads land under `FINRAG_DATA_ROOT` (default `./data`,
which is gitignored).


In [ ]:
from finrag.ingest.download import download_filings, list_filings

counts = download_filings(('AAPL', 'AMZN'), years=2, settings=settings)
print(counts)
print(f'{len(list_filings(settings=settings))} full-submission files on disk')


## 2. Index them

Each filing is parsed to text (financial tables rendered as markdown so values stay
attached to their labels), chunked, and upserted into Chroma.

**This is where the fiscal-year bug was.** The original code took the year from the
accession number, which records when a filing was *submitted*. Amazon files its FY2022
annual report in February 2023, so every chunk was labelled `year: 2023` and every
query filtered to 2023 silently returned the wrong report. Six of the ten default
tickers were affected. The year now comes from `CONFORMED PERIOD OF REPORT` in the
submission header.

Chunk IDs are deterministic, so re-running this cell updates rows in place rather
than appending a second copy of the corpus.


In [ ]:
from finrag.ingest.index import index_filings

result = index_filings(settings=settings)
print(result)


In [ ]:
# Metadata check: the period of report, not the filing date, decides the fiscal year.
from finrag.ingest.metadata import filing_metadata

for path in list_filings(settings=settings)[:4]:
    meta = filing_metadata(path)
    print(f'{meta.ticker:6} FY{meta.fiscal_year}  period={meta.period_of_report}  filed={meta.filed_as_of_date}')


## 3. Retrieval

Filtered to one company and one fiscal year, with the query expanded toward
financial-statement language before embedding.


In [ ]:
from finrag.ingest.index import open_store
from finrag.retrieval import search_filing

store = open_store(settings)
hit = search_filing('total current assets and current liabilities', 'AAPL', 2023,
                    store=store, settings=settings)
print(hit.as_context()[:1500])


## 4. The agent

Two tools: retrieval, and a calculator. The calculator evaluates arithmetic by
walking the AST and refusing every node that is not arithmetic — it replaced an
`eval()` plus Python REPL combination that executed whatever the model emitted.

Needs `GOOGLE_API_KEY`.


In [ ]:
from finrag.agent import build_agent

agent = build_agent(store=store, settings=settings, verbose=True)
answer = agent.invoke({'input': "What was Apple's current ratio in fiscal 2023?"})
print(answer['output'])


---

The same steps are available from the command line:

```bash
finrag download --tickers AAPL,AMZN --years 2
finrag index
finrag status
finrag ask "What was Apple's current ratio in fiscal 2023?"
```
